# 30. 점수 합치기 — RRF vs Min-Max (KLUE-MRC)

슬라이드 「스코어 병합 문제」 · 「RRF」 · 「Min-Max 정규화 퓨전」 · 「하이브리드 검색」

4-1 과 같은 평가: 질문 500개(`eval_q`), 정답 지문이 상위 10위 안에 오면 Hit, MRR = 정답 순위 역수의 평균.
세 가지 검색 결과(각 상위 50개)를 한 번 뽑아 두고, 합치는 방법만 바꿔 가며 비교한다.

| 이름 | 방법 |
|---|---|
| `bm25` | 4-1 에서 직접 구현한 `bm25_search()` |
| `vec_doc` | 지문 통째 임베딩 (4-1 과 같음, 앞 128토큰만 담김) |
| `vec_chunk` | 250자 청크 임베딩 → 지문별 최고 유사도 (4-2 에서 추가) |

In [ ]:
from labutil import *
import pickle

conn = connect()
log = open("prepare.log", encoding="utf-8").read().strip().splitlines() if os.path.exists("prepare.log") else []
print(log[-1] if log else "(prepare.log 없음)")
assert q1(conn, "SELECT count(*) FROM doc_chunks WHERE embedding IS NOT NULL") > 0, "00번의 KLUE 준비가 끝나야 한다"

## 1. 세 가지 검색 결과 뽑기 (약 1분)

In [ ]:
os.makedirs("results", exist_ok=True)
CACHE = "results/klue_runs.pkl"
if os.path.exists(CACHE):
    runs = pickle.load(open(CACHE, "rb"))
else:
    E = q(conn, "SELECT e.id, e.doc_id AS gold, e.question, q.embedding FROM eval_q e JOIN questions q USING (id) ORDER BY e.id")
    conn.execute("SET hnsw.ef_search = 400")
    runs = {"gold": dict(zip(E.id, E.gold)), "question": dict(zip(E.id, E.question)), "bm25": {}, "vec_doc": {}, "vec_chunk": {}}
    with timer("검색 500 × 3"):
        for qid, _, text, emb in E.itertuples(index=False):
            runs["bm25"][qid] = q(conn, "SELECT doc_id, score FROM bm25_search(%s, 50)", (text,)).values.tolist()
            runs["vec_doc"][qid] = q(conn, """SELECT id, 1 - (embedding <=> %(e)s) FROM docs
                                             ORDER BY embedding <=> %(e)s LIMIT 50""", {"e": emb}).values.tolist()
            runs["vec_chunk"][qid] = q(conn, """
                SELECT doc_id, max(sim) AS sim FROM (
                    SELECT doc_id, 1 - (embedding <=> %(e)s) AS sim FROM doc_chunks
                    ORDER BY embedding <=> %(e)s LIMIT 200) c
                GROUP BY doc_id ORDER BY sim DESC LIMIT 50""", {"e": emb}).values.tolist()
    conn.execute("RESET hnsw.ef_search")
    pickle.dump(runs, open(CACHE, "wb"))
QIDS = sorted(runs["gold"])


def evaluate(ranked, k=10):
    """ranked: qid → [doc_id, ...] (점수 높은 순). 반환: Hit@k, MRR"""
    hit, rr = [], []
    for qid in QIDS:
        lst = list(ranked[qid])[:k]
        g = runs["gold"][qid]
        r = lst.index(g) + 1 if g in lst else None
        hit.append(r is not None); rr.append(1 / r if r else 0)
    return {"Hit@10": np.mean(hit), "MRR": np.mean(rr)}


single = {m: evaluate({qid: [d for d, _ in runs[m][qid]] for qid in QIDS}) for m in ["bm25", "vec_doc", "vec_chunk"]}

fig, ax = plt.subplots(figsize=(9, 3.6))
S = pd.DataFrame(single).T
x = np.arange(len(S))
ax.bar(x - 0.2, S["Hit@10"], 0.4, color="#3D484A", label="Hit@10")
ax.bar(x + 0.2, S["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(S.values):
    ax.text(i - 0.2, h, f"{h:.3f}", ha="center", va="bottom"); ax.text(i + 0.2, m, f"{m:.3f}", ha="center", va="bottom")
ax.set_xticks(x, ["BM25", "벡터: 지문 통째", "벡터: 250자 청크"]); ax.set_ylim(0, 1.1); ax.legend()
ax.set_title("단일 검색 방법 (질문 500개)"); plt.show()

**관찰:** 청크로 나눠 임베딩하니 벡터 검색 Hit@10 이 0.39 → 0.50 으로 올랐다 (4-1 의 "지문 뒤쪽은 벡터에 안 담긴다" 문제를 일부 해결).
그래도 KLUE-MRC 는 질문이 지문의 고유명사를 그대로 쓰는 데이터라 BM25 가 강하다.

## 2. 왜 점수를 그냥 더하면 안 되나 — 척도가 다르다

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
b = np.concatenate([[s for _, s in runs["bm25"][qid][:10]] for qid in QIDS])
v = np.concatenate([[s for _, s in runs["vec_chunk"][qid][:10]] for qid in QIDS])
axes[0].hist(b, bins=60, color="#3D484A"); axes[0].set_title(f"BM25 점수 (상위 10): {b.min():.0f} ~ {b.max():.0f}")
axes[1].hist(v, bins=60, color="#FF7043"); axes[1].set_title(f"코사인 유사도 (상위 10): {v.min():.2f} ~ {v.max():.2f}")
# 한 질문의 두 목록을 한 평면에 (정답이 양쪽 목록에 다 있는 질문)
qid = next(i for i in QIDS if runs["gold"][i] in dict(runs["bm25"][i]) and runs["gold"][i] in dict(runs["vec_chunk"][i])
           and [d for d, _ in runs["vec_chunk"][i]].index(runs["gold"][i]) > 0)
bd, vd = dict(runs["bm25"][qid]), dict(runs["vec_chunk"][qid])
v_floor = min(vd.values()) - 0.03                      # 벡터 목록에 없는 문서를 그릴 높이
both = [d for d in bd if d in vd]
only_b = [d for d in bd if d not in vd]
only_v = [d for d in vd if d not in bd]
axes[2].scatter([bd[d] for d in both], [vd[d] for d in both], s=22, color="#3D484A", label="양쪽 목록에 있음")
axes[2].scatter([bd[d] for d in only_b], [v_floor] * len(only_b), s=22, marker="|", color="#FF7043", label="BM25 목록에만")
axes[2].scatter([0] * len(only_v), [vd[d] for d in only_v], s=22, marker="_", color="#009CA6", label="벡터 목록에만")
g = runs["gold"][qid]
axes[2].scatter([bd[g]], [vd[g]], s=220, marker="*", color="#F6BD60", edgecolors="black", zorder=5, label="정답 지문")
axes[2].set_xlabel("BM25 점수"); axes[2].set_ylabel("코사인 유사도"); axes[2].legend(fontsize=7)
axes[2].set_title("질문 하나의 후보 — 한쪽에만 있으면 다른 쪽 점수가 없다")
plt.tight_layout(); plt.show()

BM25 는 0~수십, 코사인은 0.3~0.9. 그냥 더하면 BM25 가 다 정한다. 게다가 **한쪽 목록에만 있는 문서**는 다른 쪽 점수가 없다.

## 3. RRF — 점수를 버리고 순위만 쓴다
`RRF(d) = Σ 1 / (k + rank)` . 슬라이드는 "k 로 변곡점을 조절"이라 했는데, 실제 곡선을 그려 보면:

In [ ]:
ranks = np.arange(1, 51)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for k, c in zip([0, 10, 60, 200], ["#FF7043", "#009CA6", "#3D484A", "#9aa5a8"]):
    w = 1 / (k + ranks)
    axes[0].plot(ranks, w / w[0], color=c, label=f"k={k}")
    axes[1].plot(ranks[:-1], np.diff(w / w[0]), color=c, label=f"k={k}")
axes[0].set_xlabel("순위"); axes[0].set_ylabel("1위 대비 가중치"); axes[0].legend(); axes[0].set_title("1/(k+rank) — k 가 작을수록 앞쪽에 쏠림")
axes[1].set_xlabel("순위"); axes[1].set_ylabel("다음 순위로 갈 때 감소량"); axes[1].set_title("감소량이 단조롭게 줄어든다 = 변곡점 없음")
plt.tight_layout(); plt.show()

쌍곡선이라 **변곡점은 없다.** k 는 "1위와 10위의 가중치 차이"(쏠림 정도)를 정한다. k=0 이면 10위가 1위의 1/10, k=60 이면 0.87.

In [ ]:
def rrf(lists, k=60, w=None):
    w = w or [1] * len(lists)
    out = {}
    for qid in QIDS:
        s = {}
        for L, wi in zip(lists, w):
            for r, (d, _) in enumerate(runs[L][qid], 1):
                s[d] = s.get(d, 0) + wi / (k + r)
        out[qid] = sorted(s, key=s.get, reverse=True)
    return out


ks = [1, 5, 10, 20, 40, 60, 100, 200]
rrf_res = pd.DataFrame([{"k": k, **evaluate(rrf(["bm25", "vec_chunk"], k))} for k in ks])
ws = [0.5, 1, 1.5, 2, 3, 5]
wrrf = pd.DataFrame([{"BM25 가중치": w, **evaluate(rrf(["bm25", "vec_chunk"], 60, [w, 1]))} for w in ws])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col, c in [("Hit@10", "#3D484A"), ("MRR", "#FF7043")]:
    axes[0].plot(rrf_res["k"], rrf_res[col], "-o", color=c, label=f"RRF {col}")
    axes[0].axhline(single["bm25"][col], color=c, ls=":", label=f"BM25 단독 {col}")
    axes[1].plot(wrrf["BM25 가중치"], wrrf[col], "-o", color=c, label=col)
axes[0].set_xscale("log"); axes[0].set_xlabel("RRF 상수 k"); axes[0].legend(fontsize=8); axes[0].set_title("BM25 + 청크 벡터, k 를 바꾸면")
axes[1].set_xlabel("BM25 쪽 가중치 (벡터 = 1, k = 60)"); axes[1].legend(); axes[1].set_title("가중 RRF")
plt.tight_layout(); plt.show()

**관찰:**
- 흔히 쓰는 k=60 은 BM25 단독보다 **MRR 이 크게 낮다**(0.84 → 0.64). 훨씬 약한 벡터 목록과 동등하게 섞기 때문 (4-1 25번과 같은 현상)
- k 를 작게(앞 순위에 쏠리게) 하거나 BM25 에 가중치를 주면 회복된다. 그래도 MRR 은 BM25 단독을 넘지 못한다

## 4. Min-Max — 점수를 0~1 로 펴서 가중 평균
`mm(s) = (s − min) / (max − min)` 를 **질문마다** 각 목록 안에서 계산. 한쪽에만 있는 문서는 다른 쪽을 0 으로 둔다.

In [ ]:
def minmax_fuse(a, b, alpha):
    out = {}
    for qid in QIDS:
        fused = {}
        for L, wgt in [(a, alpha), (b, 1 - alpha)]:
            sc = np.array([s for _, s in runs[L][qid]], dtype=float)
            lo, hi = sc.min(), sc.max()
            for d, s in runs[L][qid]:
                fused[d] = fused.get(d, 0) + wgt * ((s - lo) / (hi - lo) if hi > lo else 1)
        out[qid] = sorted(fused, key=fused.get, reverse=True)
    return out


alphas = np.round(np.linspace(0, 1, 11), 1)
mm = pd.DataFrame([{"α (BM25 비중)": a, **evaluate(minmax_fuse("bm25", "vec_chunk", a))} for a in alphas])
best_rrf = rrf_res.loc[rrf_res["MRR"].idxmax()]

fig, ax = plt.subplots(figsize=(10, 4))
for col, c in [("Hit@10", "#3D484A"), ("MRR", "#FF7043")]:
    ax.plot(mm["α (BM25 비중)"], mm[col], "-o", color=c, label=f"Min-Max {col}")
    ax.axhline(best_rrf[col], color=c, ls="--", alpha=0.6, label=f"RRF 최고 {col} (k={int(best_rrf['k'])})")
ax.set_xlabel("α : 0 = 벡터만, 1 = BM25 만"); ax.legend(fontsize=8, ncol=2); ax.set_title("Min-Max 가중 평균")
plt.show()

**관찰:** α = 0.6~0.8 에서 Hit@10 · MRR 이 모두 BM25 단독을 조금 넘는다. 이 실습에서 BM25 를 이긴 유일한 조합이다.
점수 **크기**를 살리므로, BM25 가 확신하는 질문(1위 점수가 압도적)은 그대로 두고 애매한 질문에서만 벡터가 순위를 바꾼다.
RRF 는 순위만 보므로 "얼마나 확신하는지"를 버린다.

## 5. Min-Max 의 약점 — 목록 안의 극단값 하나가 스케일을 정한다

In [ ]:
# BM25 1위가 2위보다 압도적으로 높은 질문
ratio = {qid: runs["bm25"][qid][0][1] / max(runs["bm25"][qid][1][1], 1e-9) for qid in QIDS}
qx = max(ratio, key=ratio.get)
sc = np.array([s for _, s in runs["bm25"][qx]])
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].bar(range(1, 51), sc, color="#3D484A"); axes[0].set_title(f"BM25 점수: 1위가 2위의 {ratio[qx]:.1f}배")
axes[1].bar(range(1, 51), (sc - sc.min()) / (sc.max() - sc.min()), color="#FF7043")
axes[1].set_title("Min-Max 후: 2위부터 전부 0 근처로 눌린다")
for ax in axes:
    ax.set_xlabel("순위")
plt.tight_layout(); plt.show()
print("질문:", runs["question"][qx])

2위 이하가 0 근처로 눌려서, 벡터 쪽 점수가 조금만 높아도 순위가 뒤집힌다. RRF 는 순위만 보므로 이 문제가 없다.

## 6. 종합

In [ ]:
best_mm = mm.loc[mm["MRR"].idxmax()]
best_w = wrrf.loc[wrrf["MRR"].idxmax()]
summary = pd.DataFrame({
    "BM25": single["bm25"], "벡터(지문)": single["vec_doc"], "벡터(청크)": single["vec_chunk"],
    "RRF k=60": evaluate(rrf(["bm25", "vec_chunk"], 60)),
    f"RRF 최고 (k={int(best_rrf['k'])})": best_rrf[["Hit@10", "MRR"]].to_dict(),
    f"가중 RRF (BM25×{best_w['BM25 가중치']})": best_w[["Hit@10", "MRR"]].to_dict(),
    f"Min-Max 최고 (α={best_mm['α (BM25 비중)']})": best_mm[["Hit@10", "MRR"]].to_dict(),
    "RRF 3종 (BM25+지문+청크)": evaluate(rrf(["bm25", "vec_doc", "vec_chunk"], 60)),
}).T.astype(float)

fig, ax = plt.subplots(figsize=(12, 4.6))
y = np.arange(len(summary))
ax.barh(y + 0.2, summary["Hit@10"], 0.4, color="#3D484A", label="Hit@10")
ax.barh(y - 0.2, summary["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(summary.values):
    ax.text(h, i + 0.2, f" {h:.3f}", va="center", fontsize=8); ax.text(m, i - 0.2, f" {m:.3f}", va="center", fontsize=8)
ax.set_yticks(y, summary.index); ax.invert_yaxis(); ax.set_xlim(0, 1.1); ax.legend(loc="lower right")
ax.set_title("질문 500개 종합"); plt.tight_layout(); plt.show()
summary.round(3)

**생각해 볼 것**
- RRF(k=60)는 두 목록을 **동등하게** 섞는다. 한쪽이 확실히 강하면(여기선 BM25) 가중치를 주거나 k 를 조정해야 이득이 난다
- Min-Max 는 점수 분포를 그대로 살리므로 α 를 잘 고르면 강하지만, α 는 **데이터마다 다시 찾아야 하는 값**이다
- "가장 좋은 설정"은 이 500문항에 맞춘 것이다. 실제로는 평가셋을 나눠 한쪽에서 고르고 다른 쪽에서 확인해야 과적합을 피한다
- 슬라이드: 점수 분포가 믿을 만하면 Min-Max, 제멋대로면 RRF